### Baseline model som endast använder sig av movies-datasetet och dess genrer för att ge film-tips.

In [86]:
from data_loader import load_data
import pandas as pd

movies, _, _ = load_data()

In [87]:
movies.head(), movies.shape

(   movieId                               title  \
 0        1                    Toy Story (1995)   
 1        2                      Jumanji (1995)   
 2        3             Grumpier Old Men (1995)   
 3        4            Waiting to Exhale (1995)   
 4        5  Father of the Bride Part II (1995)   
 
                                         genres  
 0  Adventure|Animation|Children|Comedy|Fantasy  
 1                   Adventure|Children|Fantasy  
 2                               Comedy|Romance  
 3                         Comedy|Drama|Romance  
 4                                       Comedy  ,
 (86537, 3))

In [88]:
genre_matrix = movies['genres'].str.get_dummies(sep='|')

genre_matrix.shape, genre_matrix.head(), genre_matrix.columns

((86537, 20),
    (no genres listed)  Action  Adventure  Animation  Children  Comedy  Crime  \
 0                   0       0          1          1         1       1      0   
 1                   0       0          1          0         1       0      0   
 2                   0       0          0          0         0       1      0   
 3                   0       0          0          0         0       1      0   
 4                   0       0          0          0         0       1      0   
 
    Documentary  Drama  Fantasy  Film-Noir  Horror  IMAX  Musical  Mystery  \
 0            0      0        1          0       0     0        0        0   
 1            0      0        1          0       0     0        0        0   
 2            0      0        0          0       0     0        0        0   
 3            0      1        0          0       0     0        0        0   
 4            0      0        0          0       0     0        0        0   
 
    Romance  Sci-Fi  Thrille

In [89]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
from IPython.display import clear_output

def recommend_movies(movies, genre_matrix, n=5):
    movie_title = input("Vilken film vill du se? ").strip()

    movie_match = movies[
        movies["title"].str.lower().str.contains(movie_title.lower(), na=False)
    ]

    if movie_match.empty:
        print("Filmen hittades inte. Kontrollera stavningen eller välj en annan film.")
        return

    if len(movie_match) > 1:
        print("Flera filmer matchade din sökning:")

        movie_options = movie_match[["title"]].reset_index()
        movie_options.index = movie_options.index + 1
        print(movie_options[["title"]])

        choice = int(input("Välj film genom att skriva numret: "))
        chosen_row = movie_options.iloc[choice - 1]
        movie_index = chosen_row["index"]
        chosen_title = chosen_row["title"]

        clear_output(wait=True)

    else:
        movie_index = movie_match.index[0]
        chosen_title = movie_match.iloc[0]["title"]

    movie_vector = genre_matrix.iloc[movie_index:movie_index + 1]
    similarity_scores = cosine_similarity(genre_matrix, movie_vector)

    similarity_df = pd.DataFrame({
        "title": movies["title"],
        "genres": movies["genres"],
        "similarity": similarity_scores.flatten()
    })

    similarity_df = similarity_df.drop(index=movie_index)

    recommendations = similarity_df.sort_values("similarity", ascending=False).head(n)

    print(f"\nOm du gillar '{chosen_title}' kanske du också gillar:")

    return recommendations


In [90]:
recommend_movies(movies, genre_matrix)


Om du gillar 'Toy Story (1995)' kanske du också gillar:


,title,genres,similarity
20025,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0
22371,"Boxtrolls, The (2014)",Adventure|Animation|Children|Comedy|Fantasy,1.0
63162,Frozen II (2019),Adventure|Animation|Children|Comedy|Fantasy,1.0
2203,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
12970,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0


In [91]:
recommend_movies(movies, genre_matrix)


Om du gillar 'James Bond (2015)' kanske du också gillar:


,title,genres,similarity
29974,The Passenger (2005),(no genres listed),1.0
41084,Revenge (1964),(no genres listed),1.0
32255,Antonio Gramsci: The Days of Prison (1977),(no genres listed),1.0
34167,L'étudiante et Monsieur Henri (2015),(no genres listed),1.0
38196,Seus Problemas Acabaram!!! (2006),(no genres listed),1.0


In [97]:
recommend_movies(movies, genre_matrix)


Om du gillar 'Moana (2016)' kanske du också gillar:


,title,genres,similarity
60343,Missing Link (2019),Adventure|Animation|Children|Comedy|Fantasy,1.0
3654,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1.0
83831,My Father's Dragon (2022),Adventure|Animation|Children|Comedy|Fantasy,1.0
30664,Scooby-Doo! Mask of the Blue Falcon (2012),Adventure|Animation|Children|Comedy|Fantasy,1.0
